In [1]:
import sys
print(sys.version)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
import torch

In [3]:
GPU = torch.cuda.get_device_name(0)
print(f"CUDA GPU: {GPU}")

CUDA GPU: NVIDIA GeForce RTX 5090


In [4]:
file_path = "kaggle/test.txt"
f = open(file_path)
print(f.read())

the file is read


In [5]:
!pip install ultralytics

import os
import shutil
import random
from sklearn.model_selection import train_test_split
import numpy as np
# Set random seed for reproducibility
random.seed(42)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


 [code] {"execution":{"iopub.status.busy":"2025-06-25T20:39:25.782008Z","iopub.execute_input":"2025-06-25T20:39:25.782319Z","iopub.status.idle":"2025-06-25T20:39:29.245635Z","shell.execute_reply.started":"2025-06-25T20:39:25.782295Z","shell.execute_reply":"2025-06-25T20:39:29.245020Z"}}

In [6]:
# ====================
# DATASET PREPARATION
# ====================

# Set paths (modify these according to your Kaggle dataset)
dataset_path = 'kaggle/beedatasetv2/train'  # Update this
output_path = 'kaggle/working/dataset'

# Create directory structure
os.makedirs(f'{output_path}/train/images', exist_ok=True)
os.makedirs(f'{output_path}/train/labels', exist_ok=True)
os.makedirs(f'{output_path}/val/images', exist_ok=True)
os.makedirs(f'{output_path}/val/labels', exist_ok=True)
os.makedirs(f'{output_path}/test/images', exist_ok=True)
os.makedirs(f'{output_path}/test/labels', exist_ok=True)

# Get list of all image files (without extension)
all_images = [os.path.splitext(f)[0] for f in os.listdir(f'{dataset_path}/images') 
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# Verify corresponding labels exist
valid_images = []
for img_base in all_images:
    label_path = f'{dataset_path}/labels/{img_base}.txt'
    if os.path.exists(label_path):
        # Verify label file is not empty
        if os.path.getsize(label_path) > 0:
            valid_images.append(img_base)
        else:
            print(f"Warning: Empty label file for {img_base}")
    else:
        print(f"Warning: Missing label for {img_base}")

print(f"Found {len(valid_images)} valid image-label pairs")

# Split dataset (80% train, 15% val, 5% test)
train_val, test = train_test_split(valid_images, test_size=0.05, random_state=42)
train, val = train_test_split(train_val, test_size=0.157, random_state=42)  # 0.157 of 95% = 15% of total

# Function to copy files with proper extensions
def copy_files(file_bases, source_img_dir, source_lbl_dir, dest_img_dir, dest_lbl_dir):
    for base in file_bases:
        # Find the actual image extension
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png']:
            if os.path.exists(f'{source_img_dir}/{base}{ext}'):
                img_path = f'{source_img_dir}/{base}{ext}'
                break
        
        if not img_path:
            print(f"Warning: Image {base} not found with common extensions")
            continue
            
        # Copy image
        shutil.copy(img_path, f'{dest_img_dir}/{os.path.basename(img_path)}')
        
        # Copy label
        label_file = f'{base}.txt'
        shutil.copy(f'{source_lbl_dir}/{label_file}', 
                   f'{dest_lbl_dir}/{label_file}')

# Copy files to their new locations
copy_files(train, 
          f'{dataset_path}/images',
          f'{dataset_path}/labels',
          f'{output_path}/train/images',
          f'{output_path}/train/labels')

copy_files(val,
          f'{dataset_path}/images',
          f'{dataset_path}/labels',
          f'{output_path}/val/images',
          f'{output_path}/val/labels')

copy_files(test,
          f'{dataset_path}/images',
          f'{dataset_path}/labels',
          f'{output_path}/test/images',
          f'{output_path}/test/labels')

print("\nDataset split complete:")
print(f"- Training: {len(train)} images")
print(f"- Validation: {len(val)} images")
print(f"- Test: {len(test)} images")

Found 18390 valid image-label pairs

Dataset split complete:
- Training: 14727 images
- Validation: 2743 images
- Test: 920 images


 [code] {"execution":{"iopub.status.busy":"2025-06-25T20:39:32.373001Z","iopub.execute_input":"2025-06-25T20:39:32.373611Z","iopub.status.idle":"2025-06-25T20:39:37.693746Z","shell.execute_reply.started":"2025-06-25T20:39:32.373589Z","shell.execute_reply":"2025-06-25T20:39:37.693040Z"}}

In [7]:
# ====================================
# COMPRESS IMAGES TO LOWER THE QUALITY
# ====================================
import cv2

def compress_and_save_images(file_bases, source_img_dir, source_lbl_dir, dest_img_dir, dest_lbl_dir, quality=30):
    """
    Compress images by lowering their JPEG quality and copy corresponding labels.
    All images are saved as .jpg files regardless of original format.
    """
    for base in file_bases:
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png']:
            candidate = os.path.join(source_img_dir, base + ext)
            if os.path.exists(candidate):
                img_path = candidate
                break

        if not img_path:
            print(f"Warning: Image {base} not found with expected extensions")
            continue

        # Read the image
        img = cv2.imread(img_path)
        if img is None:
            print(f"Error reading image: {img_path}")
            continue

        # Save with reduced quality
        dest_img_file = os.path.join(dest_img_dir, base + ".jpg")
        cv2.imwrite(dest_img_file, img, [cv2.IMWRITE_JPEG_QUALITY, quality])

        # Copy label file
        label_path = os.path.join(source_lbl_dir, base + '.txt')
        if os.path.exists(label_path):
            shutil.copy(label_path, os.path.join(dest_lbl_dir, base + '.txt'))
        else:
            print(f"Warning: Label for {base} not found")

# Apply to each split
compress_and_save_images(train,
                         f'{dataset_path}/images',
                         f'{dataset_path}/labels',
                         f'{output_path}/train/images',
                         f'{output_path}/train/labels',
                         quality=10)

compress_and_save_images(val,
                         f'{dataset_path}/images',
                         f'{dataset_path}/labels',
                         f'{output_path}/val/images',
                         f'{output_path}/val/labels',
                         quality=30)

compress_and_save_images(test,
                         f'{dataset_path}/images',
                         f'{dataset_path}/labels',
                         f'{output_path}/test/images',
                         f'{output_path}/test/labels',
                         quality=100)

print("Image compression complete!")

Image compression complete!


 [code] {"execution":{"iopub.status.busy":"2025-06-25T20:39:56.860469Z","iopub.execute_input":"2025-06-25T20:39:56.861212Z","iopub.status.idle":"2025-06-25T20:39:56.986681Z","shell.execute_reply.started":"2025-06-25T20:39:56.861178Z","shell.execute_reply":"2025-06-25T20:39:56.985729Z"}}

In [8]:
# ====================
# DATA CONFIGURATION
# ====================

# Create data.yaml file
yaml_content = f"""path: {output_path}
train: train/images
val: val/images
test: test/images

# Number of classes
nc: 1

# Class names
names: ['bee']
"""

with open(f'{output_path}/data.yaml', 'w') as f:
    f.write(yaml_content)

# Verify the file was created
# !cat {output_path}/data.yaml

In [9]:
# Verify the file was created
if os.path.exists(f"{output_path}/data.yaml"):
    print("The yaml file was written successfully")

The yaml file was written successfully


 [code] {"execution":{"iopub.status.busy":"2025-06-25T20:40:03.380118Z","iopub.execute_input":"2025-06-25T20:40:03.381068Z","iopub.status.idle":"2025-06-25T22:48:23.048697Z","shell.execute_reply.started":"2025-06-25T20:40:03.381037Z","shell.execute_reply":"2025-06-25T22:48:23.047854Z"}}

In [10]:
# ====================
# MODEL TRAINING
# ====================

from ultralytics import YOLO

# Load pretrained model (will automatically adjust to single class)
model = YOLO('yolov8x.pt')  
# Train the model
results = model.train(
    data=f'{output_path}/data.yaml',
    epochs=150,
    batch=16,
    imgsz=640,
    device=0,  # use GPU
    single_cls=True,  # force single-class training
    pretrained=True,  # use pretrained weights
    optimizer='auto',  # automatically select best optimizer
    lr0=0.01,  # initial learning rate
    lrf=0.01,  # final learning rate
    momentum=0.937,  # SGD momentum
    weight_decay=0.0005,  # optimizer weight decay
    warmup_epochs=3,  # warmup epochs
    warmup_momentum=0.8,  # warmup momentum
    box=7.5,  # box loss gain
    cls=0.5,  # cls loss gain (lower for single-class)
    dfl=1.5,  # dfl loss gain
    hsv_h=0.015,  # image HSV-Hue augmentation
    hsv_s=0.7,  # image HSV-Saturation augmentation
    hsv_v=0.4,  # image HSV-Value augmentation
    degrees=10,  # image rotation
    translate=0.1,  # image translation
    scale=0.5,  # image scale
    shear=2,  # image shear
    perspective=0.001,  # image perspective
    flipud=0.5,  # image flip up-down
    fliplr=0.5,  # image flip left-right
    mosaic=1.0,  # image mosaic
    mixup=0.1,  # image mixup
    copy_paste=0.1,  # copy-paste augmentation
    erasing=0.4,  # random erasing
    crop_fraction=0.9,  # random crop fraction
    patience=20,  # early stopping patience
    project='bee_detection',
    name='yolov8n_single_class',
    exist_ok=True  # overwrite existing project
)

WARNING 'crop_fraction' is deprecated and will be removed in the future.
Ultralytics 8.4.19  Python-3.11.9 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=kaggle/working/dataset/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_single_class, nbs=64,

In [14]:
# ====================
# EVALUATION METRICS
# ====================

from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the best model
model = YOLO('runs/detect/bee_detection/yolov8n_single_class/weights/best.pt')

# Evaluate on validation set
val_results = model.val(data=f'{output_path}/data.yaml', split='val')

# Evaluate on test set
test_results = model.val(data=f'{output_path}/data.yaml', split='test')

# Extract and display metrics
def display_metrics(results, dataset_name):
    print(f"\nEvaluation Metrics for {dataset_name} Set:")
    # Extract scalar values from arrays for single-class task
    precision = results.box.p[0] if isinstance(results.box.p, (list, np.ndarray)) else results.box.p
    recall = results.box.r[0] if isinstance(results.box.r, (list, np.ndarray)) else results.box.r
    map50 = results.box.map50[0] if isinstance(results.box.map50, (list, np.ndarray)) else results.box.map50
    map5095 = results.box.map[0] if isinstance(results.box.map, (list, np.ndarray)) else results.box.map
    f1_score = results.box.f1[0] if isinstance(results.box.f1, (list, np.ndarray)) else results.box.f1
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"mAP@50: {map50:.4f}")
    print(f"mAP@50:95: {map5095:.4f}")
    print(f"F1-Score: {f1_score:.4f}")

# Display validation and test metrics
display_metrics(val_results, "Validation")
display_metrics(test_results, "Test")

# Load training metrics from results.csv
results_csv_path = 'runs/detect/bee_detection/yolov8n_single_class/results.csv'
if os.path.exists(results_csv_path):
    metrics_df = pd.read_csv(results_csv_path)
    
    # Strip whitespace from column names (common in Ultralytics CSV output)
    metrics_df.columns = metrics_df.columns.str.strip()
    
    # Rename columns to match desired metrics (based on typical Ultralytics CSV structure)
    metrics_df = metrics_df.rename(columns={
        'metrics/precision(B)': 'Precision',
        'metrics/recall(B)': 'Recall',
        'metrics/mAP50(B)': 'mAP@50',
        'metrics/mAP50-95(B)': 'mAP@50:95'
    })
    
    # Ensure 'Epoch' column exists (usually the first column)
    if 'epoch' in metrics_df.columns:
        metrics_df = metrics_df.rename(columns={'epoch': 'Epoch'})
    else:
        metrics_df['Epoch'] = range(1, len(metrics_df) + 1)
    
    # Plotting
    plt.figure(figsize=(12, 8))
    for column in ['Precision', 'Recall', 'mAP@50', 'mAP@50:95']:
        if column in metrics_df.columns:
            plt.plot(metrics_df['Epoch'], metrics_df[column], label=column)
        else:
            print(f"Warning: Column {column} not found in results.csv")
    plt.xlabel('Epoch')
    plt.ylabel('Metric Value')
    plt.title('Training Metrics Over Epochs')
    plt.legend()
    plt.grid()
    plt.show()
    # Save the plot to the output directory
    plot_path = os.path.join(output_path, 'training_metrics.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
else:
    print(f"Warning: {results_csv_path} not found. Skipping training metrics plot.")

Ultralytics 8.4.19  Python-3.11.9 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
Model summary (fused): 113 layers, 68,124,531 parameters, 0 gradients, 257.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 544.891.0 MB/s, size: 66.1 KB)
val: Scanning C:\Users\2132249\Desktop\repos\bees-knees\kaggle\working\dataset\val\labels.cache... 2743 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2743/2743  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 56, len(boxes) = 31308. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 172/172 5.8it/s 29.5s0.1ss
                   all       2743      31308      0.902      0.894      0.937      0.585
Speed: 0.9ms preprocess, 4.8ms inference, 0.0ms loss, 1.1ms postprocess per

<Figure size 1200x800 with 1 Axes>

In [16]:
import cv2
from ultralytics import YOLO
import numpy as np

# Load your trained model
model = YOLO('runs/detect/bee_detection/yolov8n_single_class/weights/best.pt')

# Video processing function
def process_video(input_path, output_path, conf_threshold=0.5):
    # Open video file
    cap = cv2.VideoCapture(input_path)
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Process each frame
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # Perform detection
        results = model.predict(frame, conf=conf_threshold, imgsz=640)
        
        # Draw bounding boxes
        for result in results:
            boxes = result.boxes.xyxy.cpu().numpy()  # Get boxes in xyxy format
            confs = result.boxes.conf.cpu().numpy()  # Get confidence scores
            cls_ids = result.boxes.cls.cpu().numpy()  # Get class IDs
            
            for box, conf, cls_id in zip(boxes, confs, cls_ids):
                x1, y1, x2, y2 = map(int, box)
                
                # Draw rectangle
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                # Put confidence text
                label = f"Bee: {conf:.2f}"
                cv2.putText(frame, label, (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        # Write frame to output
        out.write(frame)
    
    # Release resources
    cap.release()
    out.release()
    cv2.destroyAllWindows()

# Example usage (modify paths as needed)
input_video = 'kaggle/input/pest-and-bee/1_2024-09-09_180010.mp4'
output_video = 'kaggle/working/bee_detection_output.mp4'

# Process the video
process_video(input_video, output_video, conf_threshold=0.4)

# Display the output video in notebook
from IPython.display import Video
Video(output_video, embed=True)


0: 384x640 (no detections), 38.3ms
Speed: 18.1ms preprocess, 38.3ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 14.3ms
Speed: 1.1ms preprocess, 14.3ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 14.7ms
Speed: 1.1ms preprocess, 14.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 15.0ms
Speed: 1.1ms preprocess, 15.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 item, 16.2ms
Speed: 1.2ms preprocess, 16.2ms inference, 6.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 item, 14.5ms
Speed: 1.1ms preprocess, 14.5ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 item, 14.2ms
Speed: 1.1ms preprocess, 14.2ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 item, 19.3ms
Speed: 1.1ms preprocess, 19.3ms inference, 1.8ms postprocess per ima

In [24]:
from ultralytics import YOLO
from pathlib import Path
import os, shutil, json

# ====================
# MODEL SAVE + METRICS + EXPORT
# ====================

# Re-use your dataset root folder from the prep step
# output_path = 'kaggle/working/dataset'  # already defined earlier
dataset_root = Path(output_path)
data_yaml = dataset_root / "data.yaml"    # make sure you created this for YOLO
model_dir = dataset_root / "model"
model_dir.mkdir(parents=True, exist_ok=True)

# 1) Find the latest best.pt from YOLO runs
candidates = sorted(Path(".").rglob("runs/**/weights/best.pt"), key=lambda p: p.stat().st_mtime)
if not candidates:
    raise FileNotFoundError("Could not find best.pt under runs/**/weights/. "
                            "Check that training ran and produced weights.")
best_pt = candidates[-1]
print("Using best.pt:", best_pt)

# 2) Copy best.pt into your dataset/model folder
best_pt_dst = model_dir / "best.pt"
shutil.copy2(best_pt, best_pt_dst)
print("Copied best.pt to:", best_pt_dst)

# 3) Load model from the copied location (so everything is “self-contained”)
best_model = YOLO(str(best_pt_dst))

# 4) Validate to get accuracy metrics (Precision/Recall/mAP)
metrics = best_model.val(data=str(data_yaml))

metrics_dict = {
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50-95": float(metrics.box.map),
}
print("\n=== ACCURACY METRICS ===")
for k, v in metrics_dict.items():
    print(f"{k}: {v:.4f}")

# Save metrics into dataset/model folder
with open(model_dir / "metrics.json", "w") as f:
    json.dump(metrics_dict, f, indent=4)
print("Saved metrics to:", model_dir / "metrics.json")

# 5) Export ONNX into the same model folder
onnx_out = best_model.export(format="onnx")
onnx_path = Path(onnx_out)

# If Ultralytics exported somewhere else, move it into model_dir
onnx_dst = model_dir / onnx_path.name
if onnx_path.exists() and onnx_path.resolve() != onnx_dst.resolve():
    shutil.move(str(onnx_path), str(onnx_dst))

print("Saved ONNX to:", onnx_dst)
print("\nAll model artifacts are now stored in:", model_dir)

Using best.pt: runs\detect\bee_detection\yolov8n_single_class\weights\best.pt
Copied best.pt to: kaggle\working\dataset\model\best.pt
Ultralytics 8.4.19  Python-3.11.9 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
Model summary (fused): 113 layers, 68,124,531 parameters, 0 gradients, 257.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 797.6119.5 MB/s, size: 98.7 KB)
val: Scanning C:\Users\2132249\Desktop\repos\bees-knees\kaggle\working\dataset\val\labels.cache... 2743 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2743/2743  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 56, len(boxes) = 31308. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 172/172 5.9it/s 29.2s0.1ss
                   all       

C:\Users\2132249\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\utils.py:552: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  _export(


ONNX: slimming with onnxslim 0.1.87...
ONNX: export success  4.3s, saved as 'kaggle\working\dataset\model\best.onnx' (260.1 MB)

Export complete (5.8s)
Results saved to C:\Users\2132249\Desktop\repos\bees-knees\kaggle\working\dataset\model
Predict:         yolo predict task=detect model=kaggle\working\dataset\model\best.onnx imgsz=640 
Validate:        yolo val task=detect model=kaggle\working\dataset\model\best.onnx imgsz=640 data=kaggle/working/dataset/data.yaml  
Visualize:       https://netron.app
Saved ONNX to: kaggle\working\dataset\model\best.onnx

All model artifacts are now stored in: kaggle\working\dataset\model


In [18]:
# # %% [code]
# # ====================
# # SAVE MODEL WEIGHTS
# # ====================

#Save the model weights
!cp /kaggle/working/from pathlib import Path
import os
from IPython.display import Image

test_images_dir = Path(output_path) / "test" / "images"
imgs = sorted([p for p in test_images_dir.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
if not imgs:
    raise FileNotFoundError(f"No images found in {test_images_dir}")

sample_image = str(imgs[0])

results = best_model.predict(
    sample_image,
    save=True,
    conf=0.2,
    iou=0.45,
    show_labels=True,
    show_conf=True
)

Image(filename=str(Path(results[0].save_dir) / Path(sample_image).name))bee_detection/yolov8n_single_class/weights/best.pt /kaggle/working/bee_detector.pt
!cp /kaggle/working/bee_detection/yolov8n_single_class/weights/best.onnx /kaggle/working/bee_detector.onnx

print("\nModel saved as:")
print("- /kaggle/working/bee_detector.pt")
print("- /kaggle/working/bee_detector.onnx")

'cp' is not recognized as an internal or external command,
operable program or batch file.



Model saved as:
- /kaggle/working/bee_detector.pt
- /kaggle/working/bee_detector.onnx


'cp' is not recognized as an internal or external command,
operable program or batch file.
